# 02 Data Preprocessing

Purpose: build the canonical cleaned module and job tables used by every downstream notebook.

Inputs:
- `data/modules.csv`
- `data/interim/jobs_raw.parquet` or `data/MyCareersFutureData/*.json`

Outputs:
- `data/interim/modules_clean.parquet`
- `data/interim/jobs_scope_audit.parquet`
- `data/interim/jobs_clean.parquet`
- `data/interim/preprocessing_summary.json`


In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.io import paths, require_files

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

from analysis.io import write_json
from analysis.preprocessing import clean_modules, load_raw_jobs, preprocess_jobs

require_files([paths.modules_csv, paths.jobs_dir])
print('Expected upstream inputs: raw modules CSV and raw jobs parquet/JSON files')


Expected upstream inputs: raw modules CSV and raw jobs parquet/JSON files


In [2]:
modules_raw = pd.read_csv(paths.modules_csv)
if paths.jobs_raw_parquet.exists():
    jobs_raw = pd.read_parquet(paths.jobs_raw_parquet)
    print(f'Loaded raw jobs from: {paths.jobs_raw_parquet}')
else:
    jobs_raw, _ = load_raw_jobs(paths.jobs_dir)
    jobs_raw.to_parquet(paths.jobs_raw_parquet, index=False)
    print(f'Parsed raw jobs and cached them to: {paths.jobs_raw_parquet}')

modules_clean = clean_modules(modules_raw)
preprocess = preprocess_jobs(jobs_raw, max_desc_words=120)

modules_clean.to_parquet(paths.modules_clean_parquet, index=False)
preprocess['jobs_scope_audit'].to_parquet(paths.jobs_scope_audit_parquet, index=False)
preprocess['jobs_clean'].to_parquet(paths.jobs_clean_parquet, index=False)
write_json(paths.interim_dir / 'preprocessing_summary.json', preprocess['summary'])

print(f'Saved cleaned modules to: {paths.modules_clean_parquet}')
print(f'Saved job scope audit to: {paths.jobs_scope_audit_parquet}')
print(f'Saved cleaned jobs to: {paths.jobs_clean_parquet}')


Loaded raw jobs from: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/interim/jobs_raw.parquet


Saved cleaned modules to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/interim/modules_clean.parquet
Saved job scope audit to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/interim/jobs_scope_audit.parquet
Saved cleaned jobs to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/interim/jobs_clean.parquet


In [3]:
jobs_scope_audit = preprocess['jobs_scope_audit']
jobs_clean = preprocess['jobs_clean']
jobs_excluded = preprocess['jobs_excluded']

print('=== Preprocessing Summary ===')
for key, value in preprocess['summary'].items():
    print(f'{key}: {value:,}')

if not jobs_excluded.empty:
    print('\nOut-of-scope breakdown:')
    display(jobs_excluded['role_scope'].value_counts().rename_axis('role_scope').reset_index(name='count'))

print('\nCleaned job sample:')
display(jobs_clean[['job_id', 'title', 'company', 'categories_str', 'skills_str']].head(10))
print('\nCleaned module sample:')
display(modules_clean[['moduleCode', 'title', 'faculty', 'department', 'is_undergrad']].head(10))


=== Preprocessing Summary ===
jobs_before_filtering: 22,720
jobs_after_description_filter: 22,720
jobs_excluded_out_of_scope: 1,848
jobs_exact_removed: 3,409
jobs_semantic_removed: 117
jobs_final: 17,346

Out-of-scope breakdown:


,role_scope,count
0,exclude_very_senior,768
1,exclude_tuition_teaching,568
2,exclude_internship,439
3,exclude_academia,73



Cleaned job sample:


,job_id,title,company,categories_str,skills_str
0,MCF-2025-1137542,Installation Technician,SUNINE TECHNOLOGIES PTE. LTD.,"Engineering, Manufacturing, Professional Servi...","Team Worker, Preventive Maintenance, Troublesh..."
1,MCF-2025-1180493,Dental Clinic Assistant,OXLEY DENTAL PTE. LTD.,"Customer Service, Environment / Health, Health...","Dentistry, Housekeeping, Interpersonal Skills,..."
2,MCF-2025-1283465,SUPERVISOR,SG UPAY TECHNOLOGY DEVELOPMENT PTE. LTD.,Wholesale Trade,"Negotiation, Lead Generation, Excellent Commun..."
3,MCF-2025-1324955,Sales Manager,IQPC WORLDWIDE PTE LTD,"Marketing / Public Relations, Sales / Retail","Market Research, Sponsorship, Announcements, A..."
4,MCF-2025-1441969,Housekeeping Supervisor,BCR EXPLORATION PTE. LTD.,General Management,"Negotiation, Front Office, Management Skills, ..."
5,MCF-2025-1451549,PROCUREMENT CUM QUANTITY SURVEYOR,ONECALL AIRCONDITIONING SOLUTIONS PTE. LTD.,"Customer Service, Design, Marketing / Public R...","Negotiation, Budgets, Supplier Performance, Qu..."
6,MCF-2025-1462264,PLATING MAINTENANCE TECHNICIAN,AMPHENOL FCI CONNECTORS SINGAPORE PTE. LTD.,Repair and Maintenance,"Preventive Maintenance, Troubleshooting, Analy..."
7,MCF-2025-1506743,FIELD ENGINEER,GYROMATIC INNOVATION PRIVATE LIMITED,Engineering,"Test Equipment, Construction, Hardware, Electr..."
8,MCF-2025-1550687,Site & Operations Coordinator – Construction E...,EA RECRUITMENT PTE. LTD.,"Building and Construction, Engineering","Microsoft Office, Microsoft Excel, Constructio..."
9,MCF-2025-1569420,Senior Technical,OXEN TECHNOLOGIES PTE. LTD.,"Building and Construction, Engineering, Inform...","Maintenance, Microsoft Works, Troubleshooting,..."



Cleaned module sample:


,moduleCode,title,faculty,department,is_undergrad
0,ABM5001,Leadership in Biomedicine,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
1,ABM5002,Advanced Biostatistics for Research,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
2,ABM5003,Biomedical Innovation & Enterprise,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
3,ABM5004,Capstone Project,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
4,ABM5101,Applied Immunology,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
5,ABM5102,Vaccine development and its modern applications,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
6,ABM5103,Advanced technologies in immune therapeutic de...,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
7,ABM5104,Microbiome-Aging-Immunity crosstalk,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
8,ABM5105,Drugs used in Infectious Diseases,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
9,ABM5106,Anticancer Therapeutics,Yong Loo Lin Sch of Medicine,NUS Medicine Dean's Office,False
